# NB11-CV: Refined BBSAR1 Model — Dec/Jan Leave-One-Winter-Out Cross-Validation

Repeats the analysis from `11_refined_model.ipynb` using a cross-validation scheme  
where each test fold contains all Dec + Jan days from one winter season.

**Model:** BetaBinomial with AR1 persistence (BBSAR1)  
- Features: `month_sin`, `doy_cos`, `persistence` (logit of previous day's frac_low)  
- Power prior from NB07v2 for seasonal features  

**Fold design:** Leave-one-winter-out  
- 5 folds: Dec2021+Jan2022, Dec2022+Jan2023, Dec2023+Jan2024, Dec2024+Jan2025, Dec2025+Jan2026  
- Training set = all data except the test fold  

**AR1 / persistence handling:**  
Persistence is the logit of the **previous day's observed** `frac_low` (if within 3 days, else 0).  
Because prediction is one-step-ahead (yesterday's value is always known before predicting today),  
this feature is computed from actual observations across the full dataset and is valid for test days.  
The first test day of each fold may inherit its persistence value from the last training-set day if within 3 days — this is correct deployment behaviour.

In [1]:
import os
os.environ['PYTENSOR_FLAGS'] = 'floatX=float64'
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import pytensor.tensor as pt
from scipy.special import expit, logit
from sklearn.metrics import brier_score_loss, roc_auc_score, log_loss
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression

DATA_DIR = '/Users/dlau/repos/fish-welfare/data/'
OUT_DIR  = '/Users/dlau/repos/fish-welfare/ModelSelection/dec-jan/'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"PyMC {pm.__version__}, ArviZ {az.__version__}")

PyMC 5.28.0, ArviZ 0.23.4


In [2]:
# Load and prepare data
tgt = pd.read_csv(DATA_DIR + 'nb04_target_features.csv', parse_dates=['date'])
tgt = tgt.sort_values('date').reset_index(drop=True)
tgt_clean = tgt[['date','month_sin','doy_cos','n_total','n_low','frac_low']].dropna()
tgt_clean = tgt_clean[tgt_clean['n_total'] >= 2].copy().reset_index(drop=True)

# Compute persistence over FULL dataset using actual observations (one-step-ahead deployment)
# For day t: persistence = logit(frac_low_{t-1}) if prev obs within 3 days, else 0
tgt_clean['persistence'] = 0.0
for i in range(1, len(tgt_clean)):
    days_gap = (tgt_clean['date'].iloc[i] - tgt_clean['date'].iloc[i-1]).days
    if days_gap <= 3:
        prev_frac = np.clip(tgt_clean['frac_low'].iloc[i-1], 0.01, 0.99)
        tgt_clean.iloc[i, tgt_clean.columns.get_loc('persistence')] = logit(prev_frac)

tgt_clean['month'] = tgt_clean['date'].dt.month
tgt_clean['year']  = tgt_clean['date'].dt.year
tgt_clean['bad_day'] = (tgt_clean['frac_low'] >= 0.3).astype(int)

print(f"Total obs: {len(tgt_clean)}")
print(f"Date range: {tgt_clean.date.min().date()} to {tgt_clean.date.max().date()}")
print(f"Dec/Jan obs: {len(tgt_clean[tgt_clean.month.isin([12,1])])}")

Total obs: 685
Date range: 2021-07-12 to 2026-01-27
Dec/Jan obs: 134


In [3]:
# Load power priors from NB07v2
try:
    src_post = pd.read_csv(DATA_DIR + 'nb07v2_source_posterior.csv').set_index('param')
    ms_mean = float(src_post.loc['month_sin', 'pymc_mean'])
    ms_std  = float(src_post.loc['month_sin', 'pymc_std'])
    dc_mean = float(src_post.loc['doy_cos',   'pymc_mean'])
    dc_std  = float(src_post.loc['doy_cos',   'pymc_std'])
    print(f"Source prior month_sin: {ms_mean:.3f} +/- {ms_std:.3f}")
    print(f"Source prior doy_cos:   {dc_mean:.3f} +/- {dc_std:.3f}")
except Exception as e:
    print(f"Using defaults: {e}")
    ms_mean, ms_std = 0.357, 0.086
    dc_mean, dc_std = 0.264, 0.085

a0 = 0.3  # power prior transfer strength
ms_prior_std = ms_std / np.sqrt(a0)
dc_prior_std = dc_std / np.sqrt(a0)
print(f"Power prior (a0={a0}) month_sin std: {ms_prior_std:.3f}")
print(f"Power prior (a0={a0}) doy_cos std:   {dc_prior_std:.3f}")

Source prior month_sin: 0.216 +/- 0.082
Source prior doy_cos:   0.282 +/- 0.085
Power prior (a0=0.3) month_sin std: 0.149
Power prior (a0=0.3) doy_cos std:   0.154


In [4]:
feat_cols = ['month_sin', 'doy_cos', 'persistence']

FOLD_DEFS = [
    {'name': 'Dec2021-Jan2022', 'dec_year': 2021, 'jan_year': 2022},
    {'name': 'Dec2022-Jan2023', 'dec_year': 2022, 'jan_year': 2023},
    {'name': 'Dec2023-Jan2024', 'dec_year': 2023, 'jan_year': 2024},
    {'name': 'Dec2024-Jan2025', 'dec_year': 2024, 'jan_year': 2025},
    {'name': 'Dec2025-Jan2026', 'dec_year': 2025, 'jan_year': 2026},
]

def is_test_day(row, fold):
    return ((row['month'] == 12 and row['year'] == fold['dec_year']) or
            (row['month'] ==  1 and row['year'] == fold['jan_year']))

for fold in FOLD_DEFS:
    mask = tgt_clean.apply(lambda r: is_test_day(r, fold), axis=1)
    fold['test_idx'] = tgt_clean.index[mask].tolist()
    fold['train_idx'] = tgt_clean.index[~mask].tolist()
    print(f"Fold {fold['name']}: train={len(fold['train_idx'])}, test={len(fold['test_idx'])}")

Fold Dec2021-Jan2022: train=663, test=22
Fold Dec2022-Jan2023: train=666, test=19
Fold Dec2023-Jan2024: train=663, test=22
Fold Dec2024-Jan2025: train=647, test=38
Fold Dec2025-Jan2026: train=652, test=33


In [5]:
def run_nb11_fold(fold, tgt_clean, feat_cols,
                   ms_mean, ms_std_prior, dc_mean, dc_std_prior, seed=42):
    """Fit BBSAR1 for one CV fold."""
    train = tgt_clean.loc[fold['train_idx']].reset_index(drop=True)
    test  = tgt_clean.loc[fold['test_idx']].reset_index(drop=True)

    # Standardize features based on training set only
    feat_mean = train[feat_cols].mean()
    feat_std  = train[feat_cols].std().replace(0, 1)

    X_train = ((train[feat_cols] - feat_mean) / feat_std).values
    X_test  = ((test[feat_cols]  - feat_mean) / feat_std).values

    n_train = train['n_total'].values.astype(int)
    k_train = train['n_low'].values.astype(int)

    # Rescale priors to standardized feature space
    ms_pm = ms_mean * feat_std['month_sin']
    ms_ps = ms_std_prior * feat_std['month_sin']
    dc_pm = dc_mean * feat_std['doy_cos']
    dc_ps = dc_std_prior * feat_std['doy_cos']

    with pm.Model() as model:
        intercept   = pm.Normal('intercept',   0, 5)
        b_month_sin = pm.Normal('b_month_sin', mu=ms_pm, sigma=ms_ps)
        b_doy_cos   = pm.Normal('b_doy_cos',   mu=dc_pm, sigma=dc_ps)
        phi  = pm.TruncatedNormal('phi', mu=0.3, sigma=0.2, lower=-0.9, upper=0.9)
        kappa = pm.Lognormal('kappa', mu=np.log(2.03), sigma=0.5)

        eta = (intercept
               + b_month_sin * X_train[:, 0]
               + b_doy_cos   * X_train[:, 1]
               + phi          * X_train[:, 2])
        mu = pm.math.sigmoid(eta)

        y_obs = pm.BetaBinomial('y_obs',
                                 alpha=mu * kappa,
                                 beta=(1 - mu) * kappa,
                                 n=n_train, observed=k_train)

        trace = pm.sample(
            draws=1000, tune=1000, chains=4,
            target_accept=0.9,
            progressbar=False,
            random_seed=seed,
            return_inferencedata=True
        )

    # Extract posterior samples
    b_ms   = trace.posterior['b_month_sin'].values.flatten()
    b_dc   = trace.posterior['b_doy_cos'].values.flatten()
    phi_s  = trace.posterior['phi'].values.flatten()
    int_s  = trace.posterior['intercept'].values.flatten()

    eta_test_samp = (int_s[:, None]
                     + b_ms[:, None]  * X_test[:, 0]
                     + b_dc[:, None]  * X_test[:, 1]
                     + phi_s[:, None] * X_test[:, 2])
    mu_test_samp  = expit(eta_test_samp)
    y_pred = mu_test_samp.mean(0)
    y_pred_lower = np.quantile(mu_test_samp, 0.05, axis=0)
    y_pred_upper = np.quantile(mu_test_samp, 0.95, axis=0)

    # Isotonic calibration fitted on train set
    eta_train_mean = (int_s.mean() + b_ms.mean() * X_train[:, 0]
                      + b_dc.mean() * X_train[:, 1] + phi_s.mean() * X_train[:, 2])
    mu_train_mean  = expit(eta_train_mean)
    ir = IsotonicRegression(out_of_bounds='clip')
    ir.fit(mu_train_mean, k_train / n_train)
    y_pred_cal = ir.predict(y_pred)

    y_true_frac = test['frac_low'].values
    y_true_bin  = test['bad_day'].values

    def safe_metrics(y_true_frac, y_true_bin, y_pred):
        # Soft Brier (MSE vs frac_low — no sklearn binary constraint)
        brier = float(np.mean((y_pred - y_true_frac) ** 2))
        auc   = roc_auc_score(y_true_bin, y_pred) if len(np.unique(y_true_bin)) > 1 else np.nan
        ll    = log_loss(y_true_bin, np.clip(y_pred, 1e-6, 1-1e-6)) if len(np.unique(y_true_bin)) > 1 else np.nan
        return brier, auc, ll

    brier, auc, ll         = safe_metrics(y_true_frac, y_true_bin, y_pred)
    brier_c, auc_c, ll_c  = safe_metrics(y_true_frac, y_true_bin, y_pred_cal)

    rhat_max = az.summary(trace)['r_hat'].max()
    ess_min  = az.summary(trace)['ess_bulk'].min()
    n_div    = int(trace.sample_stats['diverging'].sum())

    return dict(
        fold=fold['name'],
        n_train=len(train), n_test=len(test),
        brier=brier, auc=auc, log_loss=ll,
        brier_cal=brier_c, auc_cal=auc_c, log_loss_cal=ll_c,
        rhat_max=rhat_max, ess_min=ess_min, divergences=n_div,
        y_pred=y_pred, y_pred_cal=y_pred_cal,
        y_pred_lower=y_pred_lower, y_pred_upper=y_pred_upper,
        y_true_frac=y_true_frac, y_true_bin=y_true_bin,
        dates=test['date'].values
    )


cv_results_nb11 = []
for fold in FOLD_DEFS:
    print(f"\n=== Fold: {fold['name']} ===")
    res = run_nb11_fold(fold, tgt_clean, feat_cols,
                        ms_mean, ms_prior_std, dc_mean, dc_prior_std, seed=RANDOM_SEED)
    cv_results_nb11.append(res)
    print(f"  Raw:   Brier={res['brier']:.4f}  AUC={res['auc']:.4f}  LogLoss={res['log_loss']:.4f}")
    print(f"  Cal:   Brier={res['brier_cal']:.4f}  AUC={res['auc_cal']:.4f}  LogLoss={res['log_loss_cal']:.4f}")
    print(f"  R-hat_max={res['rhat_max']:.3f}  Divergences={res['divergences']}")


=== Fold: Dec2021-Jan2022 ===


Initializing NUTS using jitter+adapt_diag...


ERROR (pytensor.graph.rewriting.basic): SequentialGraphRewriter apply <pytensor.tensor.rewriting.elemwise.InplaceElemwiseOptimizer object at 0x10db57230>


ERROR (pytensor.graph.rewriting.basic): Traceback:


ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/Users/dlau/repos/fish-welfare/.venv/lib/python3.14/site-packages/pytensor/graph/features.py", line 643, in validate_
    ret = fgraph.execute_callbacks("validate")
  File "/Users/dlau/repos/fish-welfare/.venv/lib/python3.14/site-packages/pytensor/graph/fg.py", line 721, in execute_callbacks
    fn(self, *args, **kwargs)
    ~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dlau/repos/fish-welfare/.venv/lib/python3.14/site-packages/pytensor/graph/destroyhandler.py", line 677, in validate
    raise InconsistencyError("Dependency graph contains cycles")
pytensor.graph.utils.InconsistencyError: Dependency graph contains cycles

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/dlau/repos/fish-welfare/.venv/lib/python3.14/site-packages/pytensor/graph/rewriting/basic.py", line 289, in apply
    sub_prof = rewriter.apply(fgraph)
  File "/Users/dlau/re

...

  Raw:   Brier=0.0285  AUC=0.6216  LogLoss=0.1956
  Cal:   Brier=0.0299  AUC=0.5541  LogLoss=0.1921
  R-hat_max=1.000  Divergences=0

=== Fold: Dec2025-Jan2026 ===


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [intercept, b_month_sin, b_doy_cos, phi, kappa]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 2 seconds.


  Raw:   Brier=0.0242  AUC=0.6935  LogLoss=0.2488
  Cal:   Brier=0.0225  AUC=0.7097  LogLoss=0.2404
  R-hat_max=1.000  Divergences=0


In [6]:
# Build summary table
rows = [{k: v for k, v in r.items()
         if k not in ('y_pred', 'y_pred_cal', 'y_pred_lower', 'y_pred_upper',
                      'y_true_frac', 'y_true_bin', 'dates')}
        for r in cv_results_nb11]
cv_df11 = pd.DataFrame(rows)
print("=== NB11 Dec/Jan CV Results (raw) ===")
cols_show = ['fold','n_train','n_test','brier','auc','log_loss','rhat_max','ess_min','divergences']
print(cv_df11[cols_show].to_string(index=False))
print()
print("=== NB11 Dec/Jan CV Results (isotonic-calibrated) ===")
cols_cal = ['fold','n_train','n_test','brier_cal','auc_cal','log_loss_cal']
print(cv_df11[cols_cal].to_string(index=False))

# Pooled
y_pred_all     = np.concatenate([r['y_pred']      for r in cv_results_nb11])
y_pred_cal_all = np.concatenate([r['y_pred_cal']  for r in cv_results_nb11])
y_frac_all     = np.concatenate([r['y_true_frac'] for r in cv_results_nb11])
y_bin_all      = np.concatenate([r['y_true_bin']  for r in cv_results_nb11])

brier_pool     = float(np.mean((y_pred_all - y_frac_all) ** 2))
auc_pool       = roc_auc_score(y_bin_all, y_pred_all) if len(np.unique(y_bin_all)) > 1 else np.nan
ll_pool        = log_loss(y_bin_all, np.clip(y_pred_all, 1e-6, 1-1e-6)) if len(np.unique(y_bin_all)) > 1 else np.nan

brier_pool_c   = float(np.mean((y_pred_cal_all - y_frac_all) ** 2))
auc_pool_c     = roc_auc_score(y_bin_all, y_pred_cal_all) if len(np.unique(y_bin_all)) > 1 else np.nan
ll_pool_c      = log_loss(y_bin_all, np.clip(y_pred_cal_all, 1e-6, 1-1e-6)) if len(np.unique(y_bin_all)) > 1 else np.nan

print(f"\nPooled raw ({len(y_pred_all)} obs): Brier={brier_pool:.4f}  AUC={auc_pool:.4f}  LogLoss={ll_pool:.4f}")
print(f"Pooled cal ({len(y_pred_all)} obs): Brier={brier_pool_c:.4f}  AUC={auc_pool_c:.4f}  LogLoss={ll_pool_c:.4f}")

# Load baselines
try:
    baselines = pd.read_csv(DATA_DIR + 'nb09_baseline_results.csv')
    marginal_brier = baselines[baselines['model'].str.contains('Marginal')]['brier'].values[0]
    print(f"\nMarginal baseline Brier: {marginal_brier:.4f}")
    beats_raw = brier_pool   < marginal_brier
    beats_cal = brier_pool_c < marginal_brier
    print(f"NB11-CV pooled raw {'BEATS' if beats_raw else 'does NOT beat'} marginal baseline")
    print(f"NB11-CV pooled cal {'BEATS' if beats_cal else 'does NOT beat'} marginal baseline")
except Exception as e:
    print(f"Could not load baselines: {e}")

=== NB11 Dec/Jan CV Results (raw) ===
           fold  n_train  n_test    brier      auc  log_loss  rhat_max  ess_min  divergences
Dec2021-Jan2022      663      22 0.026652 0.400000  0.344271       1.0   4070.0            0
Dec2022-Jan2023      666      19 0.073723 0.700000  0.505869       1.0   3892.0            0
Dec2023-Jan2024      663      22 0.054766 0.843750  0.545856       1.0   3866.0            0
Dec2024-Jan2025      647      38 0.028460 0.621622  0.195575       1.0   3964.0            0
Dec2025-Jan2026      652      33 0.024249 0.693548  0.248757       1.0   4126.0            0

=== NB11 Dec/Jan CV Results (isotonic-calibrated) ===
           fold  n_train  n_test  brier_cal  auc_cal  log_loss_cal
Dec2021-Jan2022      663      22   0.024289 0.325000      0.351735
Dec2022-Jan2023      666      19   0.074545 0.708333      0.489162
Dec2023-Jan2024      663      22   0.055041 0.875000      0.557488
Dec2024-Jan2025      647      38   0.029889 0.554054      0.192127
Dec2025-Jan202

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

fold_names = [r['fold'] for r in cv_results_nb11]
xpos = np.arange(len(fold_names))
w = 0.35

# Brier
ax = axes[0]
ax.bar(xpos - w/2, [r['brier']     for r in cv_results_nb11], w, color='steelblue', alpha=0.8, label='Raw')
ax.bar(xpos + w/2, [r['brier_cal'] for r in cv_results_nb11], w, color='seagreen',  alpha=0.8, label='Calibrated')
ax.axhline(brier_pool,   color='blue',   linestyle='--', alpha=0.7, label=f'Pool raw={brier_pool:.4f}')
ax.axhline(brier_pool_c, color='green',  linestyle='--', alpha=0.7, label=f'Pool cal={brier_pool_c:.4f}')
try:
    ax.axhline(marginal_brier, color='orange', linestyle=':', label=f'Marginal={marginal_brier:.4f}')
except:
    pass
ax.set_xticks(xpos); ax.set_xticklabels(fold_names, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('Brier Score'); ax.set_title('NB11-CV: Brier per Fold'); ax.legend(fontsize=7)

# AUC
ax = axes[1]
ax.bar(xpos - w/2, [r['auc']     for r in cv_results_nb11], w, color='steelblue', alpha=0.8, label='Raw')
ax.bar(xpos + w/2, [r['auc_cal'] for r in cv_results_nb11], w, color='seagreen',  alpha=0.8, label='Calibrated')
ax.axhline(auc_pool,   color='blue',  linestyle='--', alpha=0.7, label=f'Pool raw={auc_pool:.4f}')
ax.axhline(auc_pool_c, color='green', linestyle='--', alpha=0.7, label=f'Pool cal={auc_pool_c:.4f}')
ax.set_xticks(xpos); ax.set_xticklabels(fold_names, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('AUC-ROC'); ax.set_title('NB11-CV: AUC per Fold'); ax.legend(fontsize=7)

# Log-loss
ax = axes[2]
ax.bar(xpos - w/2, [r['log_loss']     for r in cv_results_nb11], w, color='steelblue', alpha=0.8, label='Raw')
ax.bar(xpos + w/2, [r['log_loss_cal'] for r in cv_results_nb11], w, color='seagreen',  alpha=0.8, label='Calibrated')
ax.axhline(ll_pool,   color='blue',  linestyle='--', alpha=0.7, label=f'Pool raw={ll_pool:.4f}')
ax.axhline(ll_pool_c, color='green', linestyle='--', alpha=0.7, label=f'Pool cal={ll_pool_c:.4f}')
ax.set_xticks(xpos); ax.set_xticklabels(fold_names, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('Log-Loss'); ax.set_title('NB11-CV: Log-Loss per Fold'); ax.legend(fontsize=7)

plt.suptitle('NB11 BBSAR1 — Dec/Jan Leave-One-Winter-Out CV', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR + 'nb11_cv_metrics.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved nb11_cv_metrics.png")

Saved nb11_cv_metrics.png


In [8]:
fig, axes = plt.subplots(len(cv_results_nb11), 1, figsize=(14, 4 * len(cv_results_nb11)))

for ax, res in zip(axes, cv_results_nb11):
    xs = np.arange(len(res['dates']))
    ax.fill_between(xs, res['y_pred_lower'], res['y_pred_upper'],
                    alpha=0.25, color='steelblue', label='90% CI')
    ax.plot(xs, res['y_pred'], color='steelblue', lw=1.5, label='Posterior mean (raw)')
    ax.plot(xs, res['y_pred_cal'], color='seagreen', lw=1.5, linestyle='--', label='Calibrated')
    ax.scatter(xs, res['y_true_frac'], color='red', s=30, zorder=5, label='Observed frac_low')
    ax.axhline(0.3, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(f"Fold {res['fold']}  "
                 f"Brier raw={res['brier']:.4f} cal={res['brier_cal']:.4f}  "
                 f"AUC raw={res['auc']:.4f} cal={res['auc_cal']:.4f}")
    ax.set_ylabel('frac_low')
    ax.set_xticks(xs)
    ax.set_xticklabels([pd.Timestamp(d).strftime('%b-%d') for d in res['dates']], rotation=45, ha='right', fontsize=6)
    ax.legend(fontsize=8)

plt.suptitle('NB11-CV (BBSAR1): Predicted vs Observed — Dec/Jan test folds', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR + 'nb11_cv_predictions.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved nb11_cv_predictions.png")

Saved nb11_cv_predictions.png


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Reliability diagram (pooled)
ax = axes[0]
try:
    frac_pos, mean_pred = calibration_curve(y_bin_all, y_pred_all, n_bins=8, strategy='quantile')
    frac_pos_c, mean_pred_c = calibration_curve(y_bin_all, y_pred_cal_all, n_bins=8, strategy='quantile')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
    ax.plot(mean_pred, frac_pos, 's-', color='steelblue', label=f'Raw (Brier={brier_pool:.4f})')
    ax.plot(mean_pred_c, frac_pos_c, 'o-', color='seagreen', label=f'Calibrated (Brier={brier_pool_c:.4f})')
except Exception as e:
    ax.text(0.5, 0.5, f'Error: {e}', ha='center', transform=ax.transAxes)
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives (bad days)')
ax.set_title('Reliability Diagram (pooled Dec/Jan)')
ax.legend(fontsize=9)

# Prediction distribution
ax = axes[1]
ax.hist(y_pred_all, bins=20, alpha=0.6, color='steelblue', label='Raw predictions', density=True)
ax.hist(y_pred_cal_all, bins=20, alpha=0.6, color='seagreen', label='Calibrated', density=True)
ax.set_xlabel('Predicted probability')
ax.set_title('Prediction Distribution (pooled Dec/Jan)')
ax.legend()

plt.suptitle('NB11-CV: Calibration Assessment', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR + 'nb11_cv_calibration.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved nb11_cv_calibration.png")

Saved nb11_cv_calibration.png


In [10]:
print("=== MCMC Diagnostics per Fold ===")
print(f"{'Fold':<22} {'R-hat_max':>10} {'ESS_min':>10} {'Divergences':>12}")
print("-" * 58)
for res in cv_results_nb11:
    print(f"{res['fold']:<22} {res['rhat_max']:>10.4f} {res['ess_min']:>10.0f} {res['divergences']:>12}")

=== MCMC Diagnostics per Fold ===
Fold                    R-hat_max    ESS_min  Divergences
----------------------------------------------------------
Dec2021-Jan2022            1.0000       4070            0
Dec2022-Jan2023            1.0000       3892            0
Dec2023-Jan2024            1.0000       3866            0
Dec2024-Jan2025            1.0000       3964            0
Dec2025-Jan2026            1.0000       4126            0


In [11]:
# Load NB10 pooled results for comparison
try:
    nb10_pred = pd.read_csv(OUT_DIR + 'nb10_cv_pooled_predictions.csv')
    brier_nb10 = float(np.mean((nb10_pred['y_pred'] - nb10_pred['y_true_frac']) ** 2))
    auc_nb10   = roc_auc_score(nb10_pred['y_true_bin'], nb10_pred['y_pred'])
    ll_nb10    = log_loss(nb10_pred['y_true_bin'], np.clip(nb10_pred['y_pred'], 1e-6, 1-1e-6)) if len(nb10_pred['y_true_bin'].unique()) > 1 else np.nan
    print(f"NB10 (Joint BetaBinomial):  Brier={brier_nb10:.4f}  AUC={auc_nb10:.4f}  LogLoss={ll_nb10:.4f}")
except Exception as e:
    brier_nb10 = auc_nb10 = ll_nb10 = np.nan
    print(f"NB10 results not found: {e}")

print(f"NB11 raw (BBSAR1):          Brier={brier_pool:.4f}  AUC={auc_pool:.4f}  LogLoss={ll_pool:.4f}")
print(f"NB11 cal (BBSAR1+isotonic): Brier={brier_pool_c:.4f}  AUC={auc_pool_c:.4f}  LogLoss={ll_pool_c:.4f}")

# Load baselines
try:
    baselines = pd.read_csv(DATA_DIR + 'nb09_baseline_results.csv')
    print(f"\nBaseline models:")
    print(baselines[['model','brier','auc']].to_string(index=False))
except Exception as e:
    print(f"Baselines unavailable: {e}")

compare_rows = []
if not np.isnan(brier_nb10):
    compare_rows.append({'model': 'NB10 Joint BetaBin CV', 'brier': round(brier_nb10,4), 'auc': round(auc_nb10,4), 'log_loss': round(ll_nb10,4)})
compare_rows.append({'model': 'NB11 BBSAR1 raw CV',     'brier': round(brier_pool,4),   'auc': round(auc_pool,4),   'log_loss': round(ll_pool,4)})
compare_rows.append({'model': 'NB11 BBSAR1 cal CV',     'brier': round(brier_pool_c,4), 'auc': round(auc_pool_c,4), 'log_loss': round(ll_pool_c,4)})

comp_df = pd.DataFrame(compare_rows)
try:
    comp_df = pd.concat([baselines, comp_df], ignore_index=True)
except:
    pass
comp_df = comp_df.sort_values('brier').reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
colors_map = {'NB10': '#4CAF50', 'NB11': '#2196F3', 'B': '#FF9800'}
for ax, metric, title in zip(axes,
                              ['brier','auc','log_loss'],
                              ['Brier Score (lower better)',
                               'AUC-ROC (higher better)',
                               'Log-Loss (lower better)']):
    vals = comp_df[metric]
    cols = ['#4CAF50' if 'NB10' in m or 'NB11' in m else '#FF9800' for m in comp_df['model']]
    ax.barh(comp_df['model'], vals, color=cols, alpha=0.85)
    ax.set_title(title)

plt.suptitle('Dec/Jan CV: NB10 & NB11 vs Baselines', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR + 'nb11_cv_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved nb11_cv_comparison.png")

NB10 (Joint BetaBinomial):  Brier=0.0442  AUC=0.4390  LogLoss=0.2926
NB11 raw (BBSAR1):          Brier=0.0379  AUC=0.6880  LogLoss=0.3346
NB11 cal (BBSAR1+isotonic): Brier=0.0376  AUC=0.6644  LogLoss=0.3323

Baseline models:
                     model  brier    auc
         B1: Marginal rate 0.0408 0.5000
         B2: Seasonal rate 0.0686 0.5725
       B3: Target-only GLM 0.0412 0.6462
            B4: Pooled GLM 0.0412 0.6462
B5: Persistence (prev day) 0.0503 0.4236


Saved nb11_cv_comparison.png


In [12]:
cv_df11.to_csv(OUT_DIR + 'nb11_cv_results.csv', index=False)
pred_df = pd.DataFrame({
    'fold':       np.concatenate([[r['fold']] * len(r['y_pred']) for r in cv_results_nb11]),
    'date':       np.concatenate([r['dates'] for r in cv_results_nb11]),
    'y_pred':     y_pred_all,
    'y_pred_cal': y_pred_cal_all,
    'y_true_frac': y_frac_all,
    'y_true_bin':  y_bin_all,
})
pred_df.to_csv(OUT_DIR + 'nb11_cv_pooled_predictions.csv', index=False)
print("Saved nb11_cv_results.csv and nb11_cv_pooled_predictions.csv")

print()
print("=== NB11-CV FINAL SUMMARY ===")
print(f"Total Dec/Jan test observations: {len(y_pred_all)}")
print()
print("Raw predictions:")
print(f"  Pooled Brier:    {brier_pool:.4f}")
print(f"  Pooled AUC:      {auc_pool:.4f}")
print(f"  Pooled Log-Loss: {ll_pool:.4f}")
print(f"  Mean Brier (folds): {cv_df11['brier'].mean():.4f} +/- {cv_df11['brier'].std():.4f}")
print()
print("Calibrated predictions (isotonic regression):")
print(f"  Pooled Brier:    {brier_pool_c:.4f}")
print(f"  Pooled AUC:      {auc_pool_c:.4f}")
print(f"  Pooled Log-Loss: {ll_pool_c:.4f}")
print(f"  Mean Brier (folds): {cv_df11['brier_cal'].mean():.4f} +/- {cv_df11['brier_cal'].std():.4f}")

Saved nb11_cv_results.csv and nb11_cv_pooled_predictions.csv

=== NB11-CV FINAL SUMMARY ===
Total Dec/Jan test observations: 134

Raw predictions:
  Pooled Brier:    0.0379
  Pooled AUC:      0.6880
  Pooled Log-Loss: 0.3346
  Mean Brier (folds): 0.0416 +/- 0.0218

Calibrated predictions (isotonic regression):
  Pooled Brier:    0.0376
  Pooled AUC:      0.6644
  Pooled Log-Loss: 0.3323
  Mean Brier (folds): 0.0413 +/- 0.0227
